In [12]:
import tensorflow as tf
import tensorflow.keras as keras
from keras import layers
import deeplake as dl
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

In [3]:
# Loading the Dataset
train_ds = dl.load("hub://activeloop/visdrone-det-train")
val_ds = dl.load("hub://activeloop/visdrone-det-val")
test_ds = dl.load("hub://activeloop/visdrone-det-test-dev")
print(train_ds.summary())

\

Opening dataset in read-only mode as you don't have write permissions.


|

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/visdrone-det-train



|

...

Opening dataset in read-only mode as you don't have write permissions.


/

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/visdrone-det-test-dev



/

hub://activeloop/visdrone-det-test-dev loaded successfully.

Dataset(path='hub://activeloop/visdrone-det-train', read_only=True, tensors=['boxes', 'images', 'labels'])

 tensor      htype                 shape               dtype  compression
 -------    -------               -------             -------  ------- 
  boxes      bbox            (6471, 1:914, 4)         float32   None   
 images      image     (6471, 360:1500, 480:2000, 3)   uint8    jpeg   
 labels   class_label          (6471, 1:914)          uint32    None   
None


In [4]:
# Variables
BATCH_SIZE = 32
BATCH_SIZE_CLUSTER = 64
AUTOTUNE = tf.data.AUTOTUNE
pca_model = None
kmeans_model = None

GLOBAL_BACKBONE = keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3)
)
GLOBAL_BACKBONE.trainable = False

city_labels = ["tianjin", "hong-kong", "daqing", "ganzhou", 
               "guangzhou", "jinchang", "liuzhou", "nanjing", 
               "shaoxing", "shenyang", "nanyang", "zhangjiakou", 
               "suzhou", "xuzhou"]

In [5]:
# 1. PREPROCESSING WORKFLOWS

def image_preprocessing(image):
    image = tf.image.resize(image, (224, 224))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def detection_preprocessing(data):
    # Extracts the image and calculates separate counts for humans and cars.
    image = data['images']
    labels = data['labels']  # Integer Class IDs from VisDrone (0 to 9)
    
    # 1. Process the image
    image = image_preprocessing(image)
    
    # 2. Track Humans: Pedestrians (0) OR People (1)
    is_human_mask = tf.logical_or(tf.equal(labels, 0), tf.equal(labels, 1))
    human_count = tf.reduce_sum(tf.cast(is_human_mask, tf.float32))
    
    # 3. Track Cars: Car (3)
    is_car_mask = tf.equal(labels, 3)
    car_count = tf.reduce_sum(tf.cast(is_car_mask, tf.float32))
    
    # 4. Combine into a single multi-count vector: [human_count, car_count]
    # Squeezing elements into a 1D tensor of shape [2]
    counts_vector = tf.stack([human_count, car_count], axis=0)

    # 5. Normalize counts
    normalized_counts = counts_vector / 100
    
    return image, normalized_counts

# 2. FEATURE STREAM PIPELINES (Yields: Image Batch, Count Batch)

train_stream = (train_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE_CLUSTER)
                 .prefetch(AUTOTUNE))

val_stream = (val_ds.tensorflow()
                .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                .batch(BATCH_SIZE_CLUSTER)
                .prefetch(AUTOTUNE))

test_stream = (test_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE_CLUSTER)
                 .prefetch(AUTOTUNE))

In [7]:
# Processing City Labels
def classification_preprocessing(data_stream, is_training=False):
    global pca_model, kmeans_model, feature_extractor

    features = GLOBAL_BACKBONE.predict(data_stream, verbose=1)
    
    if is_training:
        pca_model = PCA(n_components=50, random_state=42)
        kmeans_model = MiniBatchKMeans(n_clusters=14, random_state=42, n_init=5, batch_size=2048)
        
        reduced_features = pca_model.fit_transform(features)
        labels = kmeans_model.fit_predict(reduced_features)
    else:
        reduced_features = pca_model.transform(features)
        labels = kmeans_model.predict(reduced_features)
        
    return tf.constant(labels, dtype=tf.int32)

tf_train_labels = classification_preprocessing(train_stream.map(lambda img, _ : img), is_training=True)
tf_val_labels = classification_preprocessing(val_stream.map(lambda img, _ : img), is_training=False)
tf_test_labels = classification_preprocessing(test_stream.map(lambda img, _ : img), is_training=False)

102/102 ━━━━━━━━━━━━━━━━━━━━ 314s 3s/step


c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


9/9 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step


c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


26/26 ━━━━━━━━━━━━━━━━━━━━ 73s 3s/step


c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [8]:
# Adding Pipelines

train_labels_ds = tf.data.Dataset.from_tensor_slices(tf_train_labels).batch(BATCH_SIZE_CLUSTER)
val_labels_ds = tf.data.Dataset.from_tensor_slices(tf_val_labels).batch(BATCH_SIZE_CLUSTER)
test_labels_ds = tf.data.Dataset.from_tensor_slices(tf_test_labels).batch(BATCH_SIZE_CLUSTER)

def shape_structure_outputs(img_batch, count_batch, label_batch):
    """
    Enforces static shapes for the Keras graph and structures targets 
    to match the multi-head model's output names.
    """
    # Enforce static ranks for the Keras graph builder
    img_batch.set_shape([None, 224, 224, 3])
    count_batch.set_shape([None, 2])
    label_batch.set_shape([None])   # [Batch_Size]
    
    return img_batch, {
        "city_output": label_batch,
        "count_output": count_batch     # Changed key name to match your model's counting head
    }

# Zip your feature streams (images + counts) with your clustering labels
train_pipeline = tf.data.Dataset.zip((train_stream, train_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_pipeline = tf.data.Dataset.zip((val_stream, val_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_pipeline = tf.data.Dataset.zip((test_stream, test_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

In [9]:
# Building the Model
def build_multi_head_model():
    input_tensor = layers.Input(shape=(224, 224, 3), name='input_image')

    base_model = keras.applications.MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_tensor=input_tensor
    )
    base_model.trainable = False

    shared_features = base_model.output

    # 1. Object Detection Head (Processes backbone features first)
    x_detect = layers.GlobalAveragePooling2D()(shared_features)
    x_detect = layers.Dense(128, activation='relu')(x_detect)
    count_output = layers.Dense(2, activation='relu', name="count_output")(x_detect)

    # 2. City Classification Head (Dependent on Detection)
    x_class_input = layers.GlobalAveragePooling2D()(shared_features)
    
    # Concatenate blends the spatial context with the localization features
    merged_features = layers.Concatenate()([x_class_input, x_detect])
    
    x_class = layers.Dense(256, activation='relu')(merged_features)
    x_class = layers.Dropout(0.5)(x_class)
    city_output = layers.Dense(14, activation='softmax', name="city_output")(x_class)

    model = keras.Model(inputs=input_tensor, outputs=[city_output, count_output])

    lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        alpha=0.0
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_scheduler),
        loss={
            "city_output": "sparse_categorical_crossentropy",
            "count_output": "mean_squared_error"
        },
        metrics={
            "city_output": "accuracy",
            "count_output": "mae"
        },
        loss_weights={
            "city_output": 1.0, 
            "count_output": 1.0
        }
    )
    model.fit(
        train_pipeline,
        validation_data=val_pipeline,
        epochs=5
    )
    
    return model

china_model = build_multi_head_model()

C:\Users\etito\AppData\Local\Temp\ipykernel_13872\1130605464.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


Epoch 1/5
    102/Unknown 195s 2s/step - city_output_accuracy: 0.4974 - city_output_loss: 1.5743 - count_output_loss: 0.0437 - count_output_mae: 0.0870 - loss: 1.6180

c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


102/102 ━━━━━━━━━━━━━━━━━━━━ 211s 2s/step - city_output_accuracy: 0.6438 - city_output_loss: 1.0236 - count_output_loss: 0.0361 - count_output_mae: 0.0782 - loss: 1.0739 - val_city_output_accuracy: 0.8376 - val_city_output_loss: 0.4266 - val_count_output_loss: 0.0352 - val_count_output_mae: 0.1050 - val_loss: 0.5143
Epoch 2/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - city_output_accuracy: 0.8050 - city_output_loss: 0.5184 - count_output_loss: 0.0345 - count_output_mae: 0.0762 - loss: 0.5609 - val_city_output_accuracy: 0.8887 - val_city_output_loss: 0.3082 - val_count_output_loss: 0.0352 - val_count_output_mae: 0.1050 - val_loss: 0.3817
Epoch 3/5
102/102 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - city_output_accuracy: 0.8493 - city_output_loss: 0.3994 - count_output_loss: 0.0345 - count_output_mae: 0.0762 - loss: 0.4418 - val_city_output_accuracy: 0.8869 - val_city_output_loss: 0.2907 - val_count_output_loss: 0.0352 - val_count_output_mae: 0.1050 - val_loss: 0.3608
Epoch 4/5
102/102 ━━━━━━━━

In [14]:
# Testing Model

results = china_model.evaluate(test_pipeline)

metrics_names = china_model.metrics_names
print("\nTest Metrics Breakdown:")
for name, value in zip(metrics_names, results):
    print(f"- {name}: {value:.4f}")

for test_images, test_targets in test_pipeline.take(1):
    
    # Generate predictions for this batch
    city_preds, count_preds = china_model.predict(test_images)
    
    # Extract the highest probability index
    predicted_city_indices = np.argmax(city_preds, axis=-1)
    true_city_indices = test_targets["city_output"].numpy().flatten()
    true_counts = test_targets["count_output"].numpy()
    
    print("\nBatch Sample Breakdown:")
    for i in range(len(test_images)):
        # Map integer indices back to string labels
        pred_label = city_labels[predicted_city_indices[i]]
        true_label = city_labels[true_city_indices[i]]
        
        print(f"\nSample {i+1}:")
        print(f"  [City] Predicted: {pred_label:<12} | True: {true_label}")
        print(f"  [Count] Predicted: {count_preds[i]*100} | True: {true_counts[i]*100}")

26/26 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - city_output_accuracy: 0.8652 - city_output_loss: 0.3281 - count_output_loss: 0.0573 - count_output_mae: 0.0761 - loss: 0.4089

Test Metrics Breakdown:
- loss: 0.4089
- compile_metrics: 0.3281
- city_output_loss: 0.0573
- count_output_loss: 0.8652
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 475ms/step

Batch Sample Breakdown:

Sample 1:
  [City] Predicted: guangzhou    | True: guangzhou
  [Count] Predicted: [0. 0.] | True: [0. 0.]

Sample 2:
  [City] Predicted: nanjing      | True: nanjing
  [Count] Predicted: [0. 0.] | True: [0. 0.]

Sample 3:
  [City] Predicted: hong-kong    | True: hong-kong
  [Count] Predicted: [0. 0.] | True: [11. 29.]

Sample 4:
  [City] Predicted: guangzhou    | True: guangzhou
  [Count] Predicted: [0. 0.] | True: [10.  0.]

Sample 5:
  [City] Predicted: shaoxing     | True: shaoxing
  [Count] Predicted: [0. 0.] | True: [0. 0.]

Sample 6:
  [City] Predicted: guangzhou    | True: guangzhou
  [Count] Predicted: [0. 0.] | True: [6. 2.]

Samp